In [ ]:
%load_ext autoreload
%autoreload 2

### Multi-dataset updating with PepCL

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name not in ("CLSBN26", "PepCL_Manuscript"):    
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find CLSBN26 project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "Data"
FIGURES_DIR = PROJECT_ROOT / "Figures"
SRC_DIR = PROJECT_ROOT / "src"
MODEL_CHECKPOINTS_DIR = PROJECT_ROOT / "Model_Checkpoints"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("FIGURES_DIR:", FIGURES_DIR)
print("PROJECT_ROOT in sys.path:", str(PROJECT_ROOT) in sys.path)

PROJECT_ROOT: /home/pchati/PepCL_Manuscript
DATA_DIR: /home/pchati/PepCL_Manuscript/Data
FIGURES_DIR: /home/pchati/PepCL_Manuscript/Figures
PROJECT_ROOT in sys.path: True


In [ ]:
# imports
from src.general_utils import *
from src.get_feature_data import *
from src.Model.mhcprime import *
from src.model_utils import *
from src.general_utils import *
from src.get_feature_data import *
from src.pepcl import *
from src.pepcl_multi import *
from src.mhcprime_inference import *
from src.plotting import *
from src.multi_dataset_updating_utils import *

# misc
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# change gpu
# GPU_ID = 0
# GPU_ID = 1
GPU_ID = 2
# GPU_ID = 3

torch.cuda.set_device(GPU_ID)
print(f"Using GPU: {torch.cuda.get_device_name(GPU_ID)} (ID: {torch.cuda.current_device()})")
device=f"cuda:{GPU_ID}"

Using GPU: NVIDIA H100 80GB HBM3 (ID: 2)


### Create figure directory

In [ ]:
# create figure dir
updated_figure_5_md_updating_dir=f"{FIGURES_DIR}/Revisions/Figure_5_MD_Updating"
updated_figure_5_md_updating_data_dir=f"{updated_figure_5_md_updating_dir}/Associated_Data"
os.makedirs(updated_figure_5_md_updating_dir, exist_ok=True)
os.makedirs(updated_figure_5_md_updating_data_dir, exist_ok=True)

### Get datasets

#### Format pseudosequences and get feature data

In [ ]:
# format pseudosequences
from Data.mhc_pseudo_seqs import mhc_seq_dict
mhc_seq_dict={k.removeprefix("HLA-").replace("*","").replace(":",""):v for k,v in mhc_seq_dict.items()}

In [ ]:
# get feature table and processed features
aa_features_table_ext, descriptor_groups=load_feature_data()
processed_feature_table, corr_info = process_aa_feature_table(aa_features_table_ext)
feature_names = list(processed_feature_table.columns)  # pass to FeatureProjectionEncoder

### Full multi-dataset run output analysis

 - Outputs from the scripts are analyzed here. We use `Multi_Dataset_Updating_Regular_Script_V2.py` and `Multi_Dataset_Updating_PepCL_Multi_Script_V2.py` to get all results. Full figure plotting is in the `Figures` directory. The outputs are in `Data/Multi_Dataset_Updating_Results`.

#### Load results and compute performance

In [ ]:
%%time
# load test sets
test_sets_per_seed={}

seeds=[0, 1, 2, 3, 4]
for seed_num in tqdm(seeds):
    # use updated test and val sets
    test_data_output_dir_base=f"{DATA_DIR}/Multi_Dataset_Updating_Results/Multi_Dataset_Updating_Regular_Final/seed_{seed_num}"
    test_data_output_dir_multi=f"{DATA_DIR}/Multi_Dataset_Updating_Results/Multi_Dataset_Updating_PepCL_Multi_Final/seed_{seed_num}"

    test_sets={
        "ms_test": pd.DataFrame(),
        "yd_test": pd.DataFrame(),
        "episcan_test": pd.DataFrame(),
        "ba_test": pd.DataFrame(),
        "escape_test": pd.DataFrame(),
    }

    test_sets_base={}
    for test_set_name, _ in test_sets.items():
        print(test_set_name)
        test_data_set=pd.read_csv(f"{test_data_output_dir_base}/{test_set_name}.csv", index_col=0)
        test_sets_base[test_set_name]=test_data_set # just overwrite

    test_sets_multi={}
    for test_set_name, _ in test_sets.items():
        print(test_set_name)
        test_data_set=pd.read_csv(f"{test_data_output_dir_multi}/{test_set_name}.csv", index_col=0)
        test_sets_multi[test_set_name]=test_data_set # just overwrite

    # get merged test sets
    test_sets_merged = merge_test_set_dicts([test_sets_base, test_sets_multi])
    print(test_sets_merged.keys())

    # NOTE: REKEY FOR NOW
    test_sets_merged = {(k[:-5] if k.endswith("_test") else k): v for k, v in test_sets_merged.items()}

    test_sets_per_seed[seed_num]=test_sets_merged

  0%|          | 0/5 [00:00<?, ?it/s]

ms_test
yd_test
episcan_test
ba_test
escape_test
ms_test
yd_test
episcan_test
ba_test
escape_test



Merging: ms_test


  Dict 2: merged | rows=3,214,300 | cols=238

Merging: yd_test


  Dict 2: merged | rows=160,000 | cols=238

Merging: episcan_test


...

  Dict 2: merged | rows=3,214,300 | cols=238

Merging: yd_test


  Dict 2: merged | rows=160,000 | cols=238

Merging: episcan_test


  Dict 2: merged | rows=80,000 | cols=239

Merging: ba_test
  Dictionary 1: dropping 2 duplicate seq_allele rows.
  Dictionary 2: dropping 2 duplicate seq_allele rows.
  Dict 2: merged | rows=45,211 | cols=238

Merging: escape_test


100%|██████████| 5/5 [06:19<00:00, 75.97s/it]

  Dict 2: merged | rows=41,867 | cols=238

Finished merging test-set dictionaries.
dict_keys(['ms_test', 'yd_test', 'episcan_test', 'ba_test', 'escape_test'])
CPU times: user 5min 36s, sys: 44 s, total: 6min 20s
Wall time: 6min 19s


In [ ]:
# define task orders
task_orders = {
    "t1": ["yd", "episcan", "ba", "escape"],
    "t2": ["episcan", "ba", "escape", "yd"],
    "t3": ["ba", "escape", "yd", "episcan"],
    "t4": ["escape", "yd", "episcan", "ba"],
    "t5": ["escape", "ba", "episcan", "yd"],
    "t6": ["yd", "escape", "ba", "episcan"],
    "t7": ["episcan", "yd", "escape", "ba"],
    "t8": ["ba", "episcan", "yd", "escape"],
}

def rekey_task_orders(task_orders):
    return {"_".join(v): v for v in task_orders.values()}

task_orders=rekey_task_orders(task_orders)
print(task_orders)

# methods dictionary
method_names_suffixes_dict={
        "FT": "_ft", 
        "PepCL w/ Replay (MS buffer)": "_pepcl_replay_full_buffer", 
        "PepCL w/ Replay/DR (MS buffer)": "_pepcl_replay_dr_full_buffer", 
        "PepCL w/ Replay (Adaptive buffer)": "_pepcl_multi_replay_full_buffer", 
        "PepCL w/ Replay/DR (Adaptive buffer)": "_pepcl_multi_replay_dr_full_buffer",
        "PepCL w/ Replay (50% MS; Adaptive buffer)": "_pepcl_multi_replay_full_buffer_50pct_ms", 
        "PepCL w/ Replay/DR (50% MS; Adaptive buffer)": "_pepcl_multi_replay_dr_full_buffer_50pct_ms",
}

{'yd_episcan_ba_escape': ['yd', 'episcan', 'ba', 'escape'], 'episcan_ba_escape_yd': ['episcan', 'ba', 'escape', 'yd'], 'ba_escape_yd_episcan': ['ba', 'escape', 'yd', 'episcan'], 'escape_yd_episcan_ba': ['escape', 'yd', 'episcan', 'ba'], 'escape_ba_episcan_yd': ['escape', 'ba', 'episcan', 'yd'], 'yd_escape_ba_episcan': ['yd', 'escape', 'ba', 'episcan'], 'episcan_yd_escape_ba': ['episcan', 'yd', 'escape', 'ba'], 'ba_episcan_yd_escape': ['ba', 'episcan', 'yd', 'escape']}


In [ ]:
%%time
# get aps
seeded_allele_ap=compute_seeded_allele_ap(
    seeded_test_sets=test_sets_per_seed, 
    task_orders=task_orders,
    method_suffixes=method_names_suffixes_dict, 
    base_score_col="mhcprime",
    verbose=True
)

Seeds: [0, 1, 2, 3, 4]
Test sets: ['ms', 'yd', 'episcan', 'ba', 'escape']
Score columns per seed: 225


Seed 0 | ms AP: 100%|██████████| 142/142 [01:39<00:00,  1.42it/s]
Seed 0 | yd AP: 100%|██████████| 16/16 [00:05<00:00,  2.75it/s]
Seed 0 | episcan AP: 100%|██████████| 4/4 [00:02<00:00,  1.53it/s]
Seed 0 | ba AP: 100%|██████████| 61/61 [00:06<00:00,  9.45it/s]
Seed 0 | escape AP: 100%|██████████| 42/42 [00:04<00:00,  9.14it/s]
Seed 1 | ms AP: 100%|██████████| 142/142 [01:39<00:00,  1.42it/s]
Seed 1 | yd AP: 100%|██████████| 16/16 [00:05<00:00,  2.76it/s]
Seed 1 | episcan AP: 100%|██████████| 4/4 [00:02<00:00,  1.52it/s]
Seed 1 | ba AP: 100%|██████████| 61/61 [00:06<00:00,  9.75it/s]
Seed 1 | escape AP: 100%|██████████| 42/42 [00:04<00:00,  8.92it/s]
Seed 2 | ms AP: 100%|██████████| 142/142 [01:40<00:00,  1.41it/s]
Seed 2 | yd AP: 100%|██████████| 16/16 [00:05<00:00,  2.76it/s]
Seed 2 | episcan AP: 100%|██████████| 4/4 [00:02<00:00,  1.53it/s]
Seed 2 | ba AP: 100%|██████████| 61/61 [00:06<00:00,  9.07it/s]
Seed 2 | escape AP: 100%|██████████| 42/42 [00:04<00:00,  9.17it/s]
Seed 3 | ms A

ms: 142 alleles × 1121 AP columns
yd: 16 alleles × 1121 AP columns
episcan: 4 alleles × 1121 AP columns
ba: 61 alleles × 1121 AP columns
escape: 42 alleles × 1121 AP columns
CPU times: user 10min 8s, sys: 7.89 s, total: 10min 15s
Wall time: 10min 14s


In [ ]:
# save task orders and outputs for plotting
sequential_runs_outputs={
    "seeded_allele_ap": seeded_allele_ap,
    "task_orders": task_orders,
    "method_names_suffixes_dict": method_names_suffixes_dict
}

save_dict_pickle(sequential_runs_outputs, f"{DATA_DIR}/Multi_Dataset_Updating_Results/Sequential_Run_Results/sequential_runs_outputs.pkl")

#### Joint updating

In [ ]:
# load results for joint updating
full_output_root=DATA_DIR / "Multi_Dataset_Updating_Results" / "All_Dataset_Joint_Updating"
test_sets_joint_updating=load_dict_pickle(f"{full_output_root}/test_sets_joint_updating.pkl")

In [ ]:
# update naming
for df_name, df in test_sets_joint_updating.items():
    print(df_name)
    if "mhcprime" in df.columns.tolist():
        df=df.drop(["mhcprime"], axis=1)
    df.rename(columns={"mhcprime_abt": "mhcprime"}, inplace=True)
    test_sets_joint_updating[df_name]=df

ms
yd
episcan
ba
escape
cov


In [ ]:
%%time
# joint results
joint_method_prefixes={
    "FT":"model_ft_multi_v2_merged",
    "PepCL":"model_pepcl_multi_v2",
}
joint_allele_ap=compute_joint_seeded_allele_ap(
    test_sets=test_sets_joint_updating,
    method_prefixes=joint_method_prefixes,
    steps=[300,1200],
    base_score_col="mhcprime",
)


Computing joint allele AP: ms
Alleles: 142
Model columns: 20


ms | Joint AP: 100%|██████████| 142/142 [00:11<00:00, 12.78it/s]


ms: 142 alleles × 21 AP columns

Computing joint allele AP: yd
Alleles: 16
Model columns: 20


yd | Joint AP: 100%|██████████| 16/16 [00:00<00:00, 25.12it/s]


yd: 16 alleles × 21 AP columns

Computing joint allele AP: episcan
Alleles: 4
Model columns: 20


...

ba: 61 alleles × 21 AP columns

Computing joint allele AP: escape
Alleles: 42
Model columns: 20


escape | Joint AP: 100%|██████████| 42/42 [00:00<00:00, 97.56it/s]


escape: 42 alleles × 21 AP columns

Computing joint allele AP: cov
Alleles: 11
Model columns: 20


cov | Joint AP: 100%|██████████| 11/11 [00:00<00:00, 11.64it/s]

cov: 11 alleles × 21 AP columns

Done.
Output keys: ['ms', 'yd', 'episcan', 'ba', 'escape', 'cov']
CPU times: user 14.3 s, sys: 87.7 ms, total: 14.4 s
Wall time: 14.3 s


In [ ]:
# methods dictionary
method_names_suffixes_dict={
        "FT": "_ft", 
        "PepCL w/ Replay (MS buffer)": "_pepcl_replay_full_buffer", 
        "PepCL w/ Replay/DR (MS buffer)": "_pepcl_replay_dr_full_buffer", 
        "PepCL w/ Replay (Adaptive buffer)": "_pepcl_multi_replay_full_buffer", 
        "PepCL w/ Replay/DR (Adaptive buffer)": "_pepcl_multi_replay_dr_full_buffer",
        "PepCL w/ Replay (50% MS; Adaptive buffer)": "_pepcl_multi_replay_full_buffer_50pct_ms", 
        "PepCL w/ Replay/DR (50% MS; Adaptive buffer)": "_pepcl_multi_replay_dr_full_buffer_50pct_ms",
}

In [ ]:
# compare joint to final for orders
comparison_ap,comparison_replicates=(
    summarize_joint_vs_sequential_allele_ap(
        sequential_allele_ap=seeded_allele_ap,
        joint_allele_ap=joint_allele_ap,
        task_orders=task_orders,
        sequential_method_suffixes=method_names_suffixes_dict,
        joint_method_prefixes=joint_method_prefixes,
        dataset_keys=[
            "ms",
            "yd",
            "episcan",
            "ba",
            "escape",
        ],
        joint_steps=[300,1200],
        base_score_col="mhcprime",
    )
)

ms: 142 common alleles | 8 sequential orders | final stage S4
  Reference: 'mhcprime' taken only from sequential allele AP table
  Sequential FT: 8 order-level means
  Sequential PepCL w/ Replay (MS buffer): 8 order-level means
  Sequential PepCL w/ Replay/DR (MS buffer): 8 order-level means
  Sequential PepCL w/ Replay (Adaptive buffer): 8 order-level means
  Sequential PepCL w/ Replay/DR (Adaptive buffer): 8 order-level means
  Sequential PepCL w/ Replay (50% MS; Adaptive buffer): 8 order-level means
  Sequential PepCL w/ Replay/DR (50% MS; Adaptive buffer): 8 order-level means
  Joint FT, 300 steps: 5 seeds
  Joint FT, 1200 steps: 5 seeds
  Joint PepCL, 300 steps: 5 seeds
  Joint PepCL, 1200 steps: 5 seeds
yd: 16 common alleles | 8 sequential orders | final stage S4
  Reference: 'mhcprime' taken only from sequential allele AP table
  Sequential FT: 8 order-level means
  Sequential PepCL w/ Replay (MS buffer): 8 order-level means
  Sequential PepCL w/ Replay/DR (MS buffer): 8 order-l

In [ ]:
scatter_summary,scatter_replicates,scatter_raw=(
    summarize_joint_vs_sequential_map(
        seeded_allele_ap=seeded_allele_ap,
        joint_allele_ap=joint_allele_ap,
        task_orders=task_orders,

        sequential_method_suffixes={
            "FT":"_ft",
            "PepCL":
                "_pepcl_multi_replay_dr_full_buffer_50pct_ms",
        },

        joint_method_prefixes={
            "FT":"model_ft_multi_v2_merged",
            "PepCL":"model_pepcl_multi_v2",
        },

        dataset_keys=[
            "ms",
            "yd",
            "episcan",
            "ba",
            "escape",
        ],

        assay_keys=[
            "yd",
            "episcan",
            "ba",
            "escape",
        ],

        joint_steps=[
            300,
            1200,
        ],

        base_score_col="mhcprime",
    )
)

In [ ]:
# save for plotting
save_dict_pickle(scatter_summary, f"{DATA_DIR}/Multi_Dataset_Updating_Results/Other_Results/joint_updating_scatter_summary.pkl")